# Conda env sara_batteries

In [1]:

import os
import sys
sys.path.append(os.path.abspath("../02_scripts"))

import utils
import config

import pandas as pd



Loading core configurations...
GPU detected: NVIDIA GeForce GTX 1650 with 3.63 GB VRAM
TEST environment detected.

Current Session Configuration Summary:
  - Device: cuda
  - Embedding Model: 'all-mpnet-base-v2'
  - Classification LLM Model: 'RedHatAI/gemma-3-4b-it-quantized.w8a8'
  - Advanced LLM Model (Ext/Synth): 'RedHatAI/gemma-3-4b-it-quantized.w8a8'
  - Data File Path: '<PROJECT_ROOT>/01_dados/df_data_batteries_130k.parquet'



In [2]:

data = utils.get_df_data()

print(data.columns)
data.head()

# Check institutions in "authors_institution"

In [12]:
########################################
# Check for duplicate institutions in "authors_institution"

import pandas as pd
import re
import unidecode
import torch
from tqdm import tqdm
from sentence_transformers import SentenceTransformer, util

# =====================================================
# 1️⃣ Extract unique institutions
# =====================================================

print("Extracting institutions...")
institutions = (
    data["authors_institution"]
    .explode()
    .dropna()
    .unique()
)

institutions = list(institutions)
print(f"Total unique institutions: {len(institutions)}")

# =====================================================
# 2️⃣ Normalization
# =====================================================

print("Normalizing names...")

def normalize(text):
    text = text.lower()
    text = unidecode.unidecode(text)
    text = re.sub(r'\buniv\b', 'university', text)
    text = re.sub(r'\binst\b', 'institute', text)
    text = re.sub(r'\bdept\b', 'department', text)
    text = re.sub(r'[^a-z0-9 ]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

normalized_institutions = [normalize(inst) for inst in institutions]

# =====================================================
# 3️⃣ Model on GPU
# =====================================================

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

model = SentenceTransformer("all-MiniLM-L6-v2", device=device)

# =====================================================
# 4️⃣ Generate embeddings (GPU)
# =====================================================

print("Generating embeddings...")

embeddings = model.encode(
    normalized_institutions,
    batch_size=128,          # GTX 1650 handles 64-128
    show_progress_bar=True,
    convert_to_tensor=True,
    normalize_embeddings=True  # important for cosine similarity
)

# =====================================================
# 5️⃣ Efficient search (NO NxN matrix)
# =====================================================

print("Searching for similar institutions...")

threshold = 0.99      # fine-tune 0.85-0.90
top_k = 10            # number of neighbors to retrieve

similar_pairs = []

for i in tqdm(range(len(embeddings))):
    hits = util.semantic_search(
        embeddings[i].unsqueeze(0),
        embeddings,
        top_k=top_k
    )[0]

    for hit in hits:
        j = hit['corpus_id']
        score = hit['score']

        if i != j and score >= threshold:
            similar_pairs.append(
                (institutions[i], institutions[j], float(score))
            )

# =====================================================
# 6️⃣ Create DataFrame
# =====================================================

similar_df = pd.DataFrame(
    similar_pairs,
    columns=["Inst_1", "Inst_2", "Similarity"]
)

similar_df = similar_df.sort_values(
    "Similarity",
    ascending=False
).drop_duplicates()

print(f"Similar pairs found: {len(similar_df)}")

similar_df.head(50)

Extracting institutions...
Total unique institutions: 17486
Normalizing names...
Using device: cuda
Generating embeddings...


Batches: 100%|██████████| 137/137 [00:04<00:00, 31.96it/s]


Searching for similar institutions...


100%|██████████| 17486/17486 [00:15<00:00, 1152.40it/s]

Similar pairs found: 2


,Inst_1,Inst_2,Similarity
0,Tianjin Lishen Battery Joint Stock Co Ltd,Tianjin Lishen Battery Joint Stock Co LTD,1.0
1,Tianjin Lishen Battery Joint Stock Co LTD,Tianjin Lishen Battery Joint Stock Co Ltd,1.0


In [7]:
similar_df.tail(50)

,Inst_1,Inst_2,Similarity
28,Shandong Energy Grp Co Ltd,Shandong Energy Grp Co LTD,1.000000
31,Shandong Energy Grp CO LTD,Shandong Energy Grp Co LTD,1.000000
2,Tianjin Lishen Battery Joint Stock Co Ltd,Tianjin Lishen Battery Joint Stock Co LTD,1.000000
22,Shenzhen CAPCHEM Technol Co Ltd,Shenzhen Capchem Technol Co Ltd,1.000000
78,Shaanxi Univ Sci &technol,Shaanxi Univ Sci & Technol,1.000000
11,Samsung SDI Co Ltd,Samsung SDI Co LTD,1.000000
73,BYD Auto Ind Co Ltd,Byd Auto Ind Co Ltd,1.000000
10,GAC AION New Energy Automobile Co Ltd,GAC Aion New Energy Automobile Co Ltd,1.000000
4,Shaanxi Univ Sci & Technol,Shaanxi Univ Sci &technol,1.000000
90,Ansteel Beijing Res Inst Co Ltd,Ansteel Beijing Res Inst Co LTD,1.000000


In [13]:
# Replace top duplicate institutions in "authors_institution"

import pandas as pd
from collections import Counter
import re

# ==========================================================
# 1️⃣ Smart capitalization function
# ==========================================================

LOWER_WORDS = {
    "of", "and", "the", "in", "on", "at", "for",
    "de", "da", "do", "dos", "das",
    "y", "e"
}

def smart_title(text):
    if not isinstance(text, str):
        return text
    
    words = text.split()
    new_words = []
    
    for i, word in enumerate(words):
        
        # Keep acronyms that are already uppercase
        if word.isupper() and len(word) >= 2:
            new_words.append(word)
            continue
        
        # Short prepositions/conjunctions (except first word)
        if i != 0 and word.lower() in LOWER_WORDS:
            new_words.append(word.lower())
            continue
        
        new_words.append(word.capitalize())
    
    return " ".join(new_words)


# ==========================================================
# 2️⃣ Actual institution frequency
# ==========================================================

print("Calculating actual frequency of institutions...")

all_institutions = (
    data["authors_institution"]
    .explode()
    .dropna()
)

freq_counter = Counter(all_institutions)


# ==========================================================
# 3️⃣ Select top 20 duplicates
# ==========================================================

top20_duplicates = similar_df.copy()


# ==========================================================
# 4️⃣ Create replacement map
# ==========================================================

print("Creating replacement map...")

replacement_map = {}

for _, row in top20_duplicates.iterrows():
    
    inst1 = row["Inst_1"]
    inst2 = row["Inst_2"]
    
    freq1 = freq_counter.get(inst1, 0)
    freq2 = freq_counter.get(inst2, 0)
    
    # Choose the more frequent one
    if freq1 >= freq2:
        keep = inst1
        replace = inst2
    else:
        keep = inst2
        replace = inst1
    
    keep_title = smart_title(keep)
    
    replacement_map[replace] = keep_title
    replacement_map[keep] = keep_title


print("Map created with", len(replacement_map), "replacements")


# ==========================================================
# 5️⃣ Apply replacement to list column (authors_institution)
# ==========================================================

def replace_in_list(inst_list):
    if isinstance(inst_list, list):
        return [
            replacement_map.get(inst, smart_title(inst))
            for inst in inst_list
        ]
    return inst_list


data["authors_institution"] = data["authors_institution"].apply(replace_in_list)


# ==========================================================
# 6️⃣ Apply replacement to string columns
# ==========================================================

def replace_single(inst):
    if isinstance(inst, str):
        return replacement_map.get(inst, smart_title(inst))
    return inst


data["first_institution"] = data["first_institution"].apply(replace_single)
data["last_institution"] = data["last_institution"].apply(replace_single)


print("Replacement completed across all 3 columns.")

Calculating actual frequency of institutions...
Creating replacement map...
Map created with 2 replacements
Replacement completed across all 3 columns.


In [14]:
# Perform an extra duplicate check on "first_institution" and "last_institution" columns

import pandas as pd
import re
import unidecode
import torch
from tqdm import tqdm
from sentence_transformers import SentenceTransformer, util

# ==========================================================
# 1️⃣ Extract unique institutions from both columns
# ==========================================================

print("Extracting unique institutions (first + last)...")

inst_first = data["first_institution"].dropna().unique()
inst_last = data["last_institution"].dropna().unique()

institutions = list(set(inst_first).union(set(inst_last)))

print(f"Total unique institutions: {len(institutions)}")


# ==========================================================
# 2️⃣ Normalization (same as used above)
# ==========================================================

def normalize(text):
    text = text.lower()
    text = unidecode.unidecode(text)
    text = re.sub(r'\buniv\b', 'university', text)
    text = re.sub(r'\binst\b', 'institute', text)
    text = re.sub(r'\bdept\b', 'department', text)
    text = re.sub(r'[^a-z0-9 ]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

normalized_institutions = [normalize(inst) for inst in institutions]


# ==========================================================
# 3️⃣ Model on GPU
# ==========================================================

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

model = SentenceTransformer("all-MiniLM-L6-v2", device=device)


# ==========================================================
# 4️⃣ Generate embeddings
# ==========================================================

print("Generating embeddings...")

embeddings = model.encode(
    normalized_institutions,
    batch_size=128,
    show_progress_bar=True,
    convert_to_tensor=True,
    normalize_embeddings=True
)


# ==========================================================
# 5️⃣ Efficient similarity search
# ==========================================================

print("Searching for potential duplicates...")

threshold = 0.99
top_k = 10

similar_pairs = []

for i in tqdm(range(len(embeddings))):
    hits = util.semantic_search(
        embeddings[i].unsqueeze(0),
        embeddings,
        top_k=top_k
    )[0]
    
    for hit in hits:
        j = hit['corpus_id']
        score = hit['score']
        
        if i != j and score >= threshold:
            similar_pairs.append(
                (institutions[i], institutions[j], float(score))
            )


# ==========================================================
# 6️⃣ Create result DataFrame
# ==========================================================

duplicates_check_df = (
    pd.DataFrame(similar_pairs, columns=["Inst_1", "Inst_2", "Similarity"])
    .sort_values("Similarity", ascending=False)
    .drop_duplicates()
)

print(f"Potential duplicates found: {len(duplicates_check_df)}")

duplicates_check_df.head(20)

Extracting unique institutions (first + last)...
Total unique institutions: 9435
Using device: cuda
Generating embeddings...


Batches: 100%|██████████| 74/74 [00:02<00:00, 32.50it/s]


Searching for potential duplicates...


100%|██████████| 9435/9435 [00:05<00:00, 1859.99it/s]

Potential duplicates found: 0


,Inst_1,Inst_2,Similarity


# Check countries in "authors_country"

In [18]:
########################################
# Check for duplicate countries in "authors_country"

import pandas as pd
import re
import unidecode
import torch
from tqdm import tqdm
from sentence_transformers import SentenceTransformer, util

# =====================================================
# 1️⃣ Extract unique countries from list
# =====================================================

print("Extracting countries...")

countries = (
    data["authors_country"]
    .explode()
    .dropna()
    .unique()
)

countries = list(countries)

print(f"Total unique countries: {len(countries)}")


# =====================================================
# 2️⃣ Normalization (simpler than institutions)
# =====================================================

print("Normalizing country names...")

def normalize_country(text):
    if not isinstance(text, str):
        return text
    
    text_clean = text.lower().strip()
    text_clean = unidecode.unidecode(text_clean)
    text_clean = re.sub(r'[^a-z0-9 ]', '', text_clean)
    text_clean = re.sub(r'\s+', ' ', text_clean)

    # Deterministic dictionary
    country_map = {
        "united states of america": "USA",
        "united states": "USA",
        "us": "USA",
        "u s": "USA",
        "u s a": "USA",
        "usa": "USA",
        "u.s.a": "USA",
        "u.s": "USA",

        "united kingdom": "UK",
        "uk": "UK",
        "u k": "UK",

        "south korea": "South Korea",
        "republic of korea": "South Korea",

        "russian federation": "Russia",
        
        "peoples republic of china": "China",
        "pr china": "China"
    }

    if text_clean in country_map:
        return country_map[text_clean]
    
    # Otherwise use Title Case
    return text.title()

normalized_countries = [normalize_country(c) for c in countries]


# =====================================================
# 3️⃣ Model on GPU
# =====================================================

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

model = SentenceTransformer("all-MiniLM-L6-v2", device=device)


# =====================================================
# 4️⃣ Generate embeddings
# =====================================================

print("Generating embeddings...")

embeddings = model.encode(
    normalized_countries,
    batch_size=128,
    show_progress_bar=True,
    convert_to_tensor=True,
    normalize_embeddings=True
)


# =====================================================
# 5️⃣ Search for similar countries
# =====================================================

print("Searching for potentially duplicate countries...")

threshold = 0.99   # high threshold to avoid false positives
top_k = 5

similar_pairs = []

for i in tqdm(range(len(embeddings))):
    hits = util.semantic_search(
        embeddings[i].unsqueeze(0),
        embeddings,
        top_k=top_k
    )[0]

    for hit in hits:
        j = hit['corpus_id']
        score = hit['score']

        if i != j and score >= threshold:
            similar_pairs.append(
                (countries[i], countries[j], float(score))
            )


# =====================================================
# 6️⃣ Create result DataFrame
# =====================================================

similar_countries_df = (
    pd.DataFrame(similar_pairs, columns=["Country_1", "Country_2", "Similarity"])
    .sort_values("Similarity", ascending=False)
    .drop_duplicates()
)

print(f"Similar pairs found: {len(similar_countries_df)}")

similar_countries_df.head(50)

Extracting countries...
Total unique countries: 147
Normalizing country names...
Using device: cuda
Generating embeddings...


Batches: 100%|██████████| 2/2 [00:00<00:00, 76.10it/s]


Searching for potentially duplicate countries...


100%|██████████| 147/147 [00:00<00:00, 5098.80it/s]

Similar pairs found: 0


,Country_1,Country_2,Similarity


In [17]:
# Replace duplicate countries in authors_country, first_country, and last_country

import pandas as pd
from collections import Counter

# ==========================================================
# 1️⃣ Smart formatting function for countries
# ==========================================================

def smart_country(text):
    if not isinstance(text, str):
        return text
    
    text = text.strip()
    
    # Keep uppercase acronyms (2-4 letters)
    if text.isupper() and 2 <= len(text) <= 4:
        return text
    
    # Otherwise standard Title Case
    return text.title()


# ==========================================================
# 2️⃣ Actual country frequency (in main list)
# ==========================================================

print("Calculating actual frequency of countries...")

all_countries = (
    data["authors_country"]
    .explode()
    .dropna()
)

freq_counter = Counter(all_countries)


# ==========================================================
# 3️⃣ Use detected duplicates
# ==========================================================

top_duplicates = similar_countries_df.copy()


# ==========================================================
# 4️⃣ Create replacement map
# ==========================================================

print("Creating country replacement map...")

replacement_map = {}

for _, row in top_duplicates.iterrows():
    
    c1 = row["Country_1"]
    c2 = row["Country_2"]
    
    freq1 = freq_counter.get(c1, 0)
    freq2 = freq_counter.get(c2, 0)
    
    # Choose the more frequent one
    if freq1 >= freq2:
        keep = c1
        replace = c2
    else:
        keep = c2
        replace = c1
    
    keep_standard = smart_country(keep)
    
    replacement_map[replace] = keep_standard
    replacement_map[keep] = keep_standard


print("Map created with", len(replacement_map), "replacements")


# ==========================================================
# 5️⃣ Apply replacement to list column
# ==========================================================

def replace_country_list(country_list):
    if isinstance(country_list, list):
        return [
            replacement_map.get(c, smart_country(c))
            for c in country_list
        ]
    return country_list


data["authors_country"] = data["authors_country"].apply(replace_country_list)


# ==========================================================
# 6️⃣ Apply replacement to string columns
# ==========================================================

def replace_country_single(c):
    if isinstance(c, str):
        return replacement_map.get(c, smart_country(c))
    return c


data["first_country"] = data["first_country"].apply(replace_country_single)
data["last_country"] = data["last_country"].apply(replace_country_single)


print("Replacement completed across all 3 country columns.")

Calculating actual frequency of countries...
Creating country replacement map...
Map created with 22 replacements
Replacement completed across all 3 country columns.


In [21]:
import pandas as pd
import re
import unidecode
import torch
from tqdm import tqdm
from sentence_transformers import SentenceTransformer, util

# ==========================================================
# 1️⃣ Extract unique countries from both columns
# ==========================================================

print("Extracting unique countries (first + last)...")

country_first = data["first_country"].dropna().unique()
country_last = data["last_country"].dropna().unique()

countries = list(set(country_first).union(set(country_last)))

print(f"Total unique countries: {len(countries)}")


# ==========================================================
# 2️⃣ Simple normalization for comparison
# ==========================================================

def normalize_country(text):
    text = text.lower()
    text = unidecode.unidecode(text)
    text = re.sub(r'[^a-z0-9 ]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

normalized_countries = [normalize_country(c) for c in countries]


# ==========================================================
# 3️⃣ Model on GPU
# ==========================================================

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

model = SentenceTransformer("all-MiniLM-L6-v2", device=device)


# ==========================================================
# 4️⃣ Generate embeddings
# ==========================================================

print("Generating embeddings...")

embeddings = model.encode(
    normalized_countries,
    batch_size=128,
    show_progress_bar=True,
    convert_to_tensor=True,
    normalize_embeddings=True
)


# ==========================================================
# 5️⃣ Efficient similarity search
# ==========================================================

print("Searching for potential duplicates among countries...")

threshold = 0.995   # high threshold for countries
top_k = 5

similar_pairs = []

for i in tqdm(range(len(embeddings))):
    hits = util.semantic_search(
        embeddings[i].unsqueeze(0),
        embeddings,
        top_k=top_k
    )[0]
    
    for hit in hits:
        j = hit['corpus_id']
        score = hit['score']
        
        if i != j and score >= threshold:
            similar_pairs.append(
                (countries[i], countries[j], float(score))
            )


# ==========================================================
# 6️⃣ Create result DataFrame
# ==========================================================

duplicates_country_check_df = (
    pd.DataFrame(similar_pairs, columns=["Country_1", "Country_2", "Similarity"])
    .sort_values("Similarity", ascending=False)
    .drop_duplicates()
)


# ==========================================================
# 7️⃣ Final country count (first + last)
# ==========================================================

print("\nGenerating final country counts (first + last)...")

all_countries_final = pd.concat([
    data["first_country"],
    data["last_country"]
]).dropna()

country_counts = (
    all_countries_final
    .value_counts()
    .reset_index()
)

country_counts.columns = ["Country", "Count"]

print(f"\nTotal distinct countries: {country_counts.shape[0]}\n")

pd.set_option('display.max_rows', None)

print(country_counts)

# optional: reset to default afterwards
pd.reset_option('display.max_rows')


print(f"Potential duplicates found: {len(duplicates_country_check_df)}")

duplicates_country_check_df.head(20)


Extracting unique countries (first + last)...
Total unique countries: 126
Using device: cuda
Generating embeddings...


Batches: 100%|██████████| 1/1 [00:00<00:00, 160.79it/s]


Searching for potential duplicates among countries...


100%|██████████| 126/126 [00:00<00:00, 2627.74it/s]


Generating final country counts (first + last)...

Total distinct countries: 126

                              Country   Count
0                     Peoples R China  139345
1                       United States   23423
2                         South Korea   18000
3                             Germany    9466
4                               India    8630
5                               Japan    8522
6                             Unknown    4592
7                           Australia    3535
8                             England    3495
9                              France    3385
10                             Canada    3328
11                             Taiwan    3123
12                          Singapore    2844
13                              Italy    2107
14                              Spain    2097
15                             Russia    1845
16                               Iran    1538
17                             Sweden    1500
18                             Turkey    12

,Country_1,Country_2,Similarity


In [23]:
utils.save_df_data(data)
utils.export_df_to_csv(data)

Data saved to: <PROJECT_ROOT>/01_dados/df_data_batteries_130k.parquet
Data exported to CSV at: <PROJECT_ROOT>/01_dados/df_data_batteries_130k.csv
